# Case 1 — univariate wPCA on Carajas TMI

**Windowed PCA (wPCA)** learns the geophysical *geometry* of one known deposit and ranks every other window of the map by how closely it reproduces that geometry. A deposit-sized window slides across the grid; each window becomes one observation; PCA distils each window to a few components; the components most characteristic of the reference deposit carry the weight; and every window is scored by its weighted distance to the reference in PCA space. The top-ranked windows are the proposed follow-up targets, and the ranking is validated by checking whether it rediscovers *other* known deposits that were held out.

This notebook runs the paper's **Case 1**: univariate TMI, reference deposit **Paulo Afonso (Deposit 6)**, k = 17 components. It uses the current config-driven pipeline (`spatial_pca.pipeline.run_spca_from_config`), the same path as the README quickstart `scripts/run_project_from_config.py --config configs/carajas_uni_tmi.yaml`.

**Prerequisites:** the environment from `requirements.txt`, and the Carajas TMI data placed under `data/` (see the README 'Public Data Download' section). If you do not have the data yet, run `00_illustrative_synthetic_demo.ipynb` first — it needs no external data.

## 1. Set up paths

In [ ]:
import os, sys
from pathlib import Path

# This notebook lives in <repo>/notebooks/. Find the repository root and put
# the source package on the path, then work from the repo root so the config's
# relative data paths resolve.
REPO = Path.cwd()
if REPO.name == 'notebooks':
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
print('Repository root:', REPO)

## 2. Check the input data is present

In [ ]:
# The Carajas grids and deposit polygons are distributed outside GitHub (see the
# README 'Public Data Download' section). This cell checks they are in place.
data_dir = REPO / 'data' / 'Carajas_Brazil_Univariate_TMI'
if not any(data_dir.glob('*.ers')):
    print('Carajas TMI data not found under', data_dir)
    print('Download the two public Drive folders linked in the README and place them under data/.')
    print('No data yet? Start with notebooks/00_illustrative_synthetic_demo.ipynb, which needs no external data.')
else:
    print('Data found:', data_dir)

## 3. Run the univariate wPCA workflow

This takes about a minute. It writes a self-contained output folder under `outputs/` with the top-window GeoPackage, diagnostic figures, the recovery curve, the resolved config, and provenance.

In [ ]:
from spatial_pca.pipeline import run_spca_from_config

# Case 1: univariate TMI, reference deposit Paulo Afonso (Deposit 6), 17 retained
# components, top 250 windows. This is the same run as the README quickstart
#   python scripts/run_project_from_config.py --config configs/carajas_uni_tmi.yaml
results = run_spca_from_config(
    REPO / 'configs' / 'carajas_uni_tmi.yaml',
    deposit_1based=6,
    k_pcs=17,
    top_k=250,
)
res = results[0]
out = Path(res.top_windows_path).parent
print('Output folder :', out)
print('Top windows   :', res.top_windows_path)
print('Recovery plot :', res.recovery_plot_path)

## 4. Show the results

In [ ]:
from IPython.display import Image, display

# Prediction map: the top-ranked windows over the TMI grid.
for p in sorted(out.glob('*Top_*Predicted_Windows.png')):
    display(Image(filename=str(p)))

# Cumulative footprint-recovery curve vs a random selection.
rec = out / 'cumulative_footprint_recovery_fraction.png'
if rec.exists():
    display(Image(filename=str(rec)))

## Expected headline result

In the top 250 windows the run reports **47.0% cumulative footprint recovery** with **5 of 11 test deposits hit** — the paper's Case 1 headline. For the exact assertion gates (recovery, AUC 57.9, hit ranks), run the scientific-record script:

```bash
python paper/case1_uni_repro.py
```